In [ ]:
import pandas as pd
import requests
import zipfile
import io
from datetime import datetime, timedelta

# 수집할 날짜 설정 (예: 어제)
target_date = datetime.utcnow() - timedelta(days=1)
date_str = target_date.strftime('%Y%m%d')

# GDELT 2.0 Events 데이터 ZIP URL
url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
print(f"Downloading: {url}")

# ZIP 다운로드 및 CSV 로딩
response = requests.get(url)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_filename = z.namelist()[0]
        with z.open(csv_filename) as f:
            df = pd.read_csv(f, sep='\t', header=None, low_memory=False)
            print("데이터 로드 완료")
else:
    raise Exception(f"다운로드 실패: {response.status_code}")

# GDELT 2.0 Events 데이터의 58개 컬럼명
cols = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources", "NumArticles",
    "AvgTone", "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_Lat", "Actor1Geo_Long", "Actor1Geo_FeatureID",
    "Actor2Geo_Type", "Actor2Geo_FullName", "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code", "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID",
    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_Lat", "ActionGeo_Long", "ActionGeo_FeatureID",
    "DATEADDED", "SOURCEURL"
]

df.columns = cols

# 필터링 기준 키워드
keywords = ['nvidia', 'apple', 'amazon', 'microsoft', 'google', 'alphabet']

# Actor1 또는 Actor2 이름에 키워드 포함 여부로 필터링 (대소문자 무시)
df_companies = df[
    df['Actor1Name'].fillna('').str.lower().str.contains('|'.join(keywords)) |
    df['Actor2Name'].fillna('').str.lower().str.contains('|'.join(keywords))
]

print(f"기업 관련 이벤트 수: {len(df_companies)}")

# 주요 컬럼 확인
df_companies[[
    "SQLDATE", "DATEADDED", "GLOBALEVENTID", "SOURCEURL",
    "Actor1Name", "Actor2Name", "Actor1CountryCode", "Actor2CountryCode",
    "Actor1Type1Code", "Actor2Type1Code",
    "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "AvgTone",
    "ActionGeo_CountryCode", "ActionGeo_Lat", "ActionGeo_Long"
]].head()


C:\Users\james\AppData\Local\Temp\ipykernel_25584\297201090.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_date = datetime.utcnow() - timedelta(days=1)


Downloading: http://data.gdeltproject.org/events/20250729.export.CSV.zip
데이터 로드 완료
기업 관련 이벤트 수: 538


,SQLDATE,Actor1Name,Actor2Name,EventCode,GoldsteinScale,AvgTone,SOURCEURL
1641,20250729,NVIDIA,NaN,10,0.0,1.666667,https://www.whitecountycitizen.com/business/ma...
1644,20250729,MICROSOFT,LOS ANGELES,42,1.9,0.928793,https://www.elliotlakestandard.ca:443/entertai...
2440,20250729,UNITED STATES,NVIDIA,10,0.0,1.666667,https://www.whitecountycitizen.com/business/ma...
2441,20250729,LOS ANGELES,MICROSOFT,43,2.8,0.928793,https://www.elliotlakestandard.ca:443/entertai...
5742,20250729,WEBSITE,GOOGLE,10,0.0,1.158111,https://www.funkytaurusmedia.com/shop_content....
